# model_sanity_check

In [5]:
# This cell sets up a reusable environment for testing any CNN version:
# 1. Adds project root to Python path
# 2. Imports torch and model classes from src
# 3. Ensures this notebook works for CNN_V1, V2, V3 without modification

import sys
import os

sys.path.append("..")

import torch
from src.models import CNN_V1

In [2]:
# This cell diagnoses import and path issues:
# 1. Confirms current working directory
# 2. Confirms Python path
# 3. Attempts safe import of CNN_V1

import os
import sys

print("Current working directory:", os.getcwd())
print("Python path:", sys.path)

sys.path.append("..")

try:
    from src.models import CNN_V1
    model = CNN_V1()
    print("CNN_V1 imported successfully")
    print(model)

except Exception as e:
    print("ERROR DURING IMPORT OR INITIALIZATION:")
    print(e)

Current working directory: d:\ML-PROJECTS\Cifar_10_Computer_Vision_Project_System\notebooks
Python path: ['c:\\Users\\User\\anaconda3\\envs\\cifar10\\python310.zip', 'c:\\Users\\User\\anaconda3\\envs\\cifar10\\DLLs', 'c:\\Users\\User\\anaconda3\\envs\\cifar10\\lib', 'c:\\Users\\User\\anaconda3\\envs\\cifar10', '', 'c:\\Users\\User\\anaconda3\\envs\\cifar10\\lib\\site-packages', 'c:\\Users\\User\\anaconda3\\envs\\cifar10\\lib\\site-packages\\win32', 'c:\\Users\\User\\anaconda3\\envs\\cifar10\\lib\\site-packages\\win32\\lib', 'c:\\Users\\User\\anaconda3\\envs\\cifar10\\lib\\site-packages\\Pythonwin', '..']
CNN_V1 imported successfully
CNN_V1(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilat

In [3]:
# This cell:
# 1. Instantiates CNN_V1 model
# 2. Verifies architecture loads correctly
# 3. Prints model structure for validation

model = CNN_V1()

print("CNN_V1 successfully created")
print(model)

CNN_V1 successfully created
CNN_V1(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=10, bias=True)
  )
)


In [4]:
# This cell:
# 1. Creates a dummy CIFAR-10 input tensor
# 2. Runs a forward pass through CNN_V1
# 3. Validates output shape matches (batch_size, 10)

x = torch.randn(4, 3, 32, 32)  # batch of 4 fake images

output = model(x)

print("Output shape:", output.shape)  # expected: torch.Size([4, 10])

Output shape: torch.Size([4, 10])


In [7]:
# This cell:
# 1. Recreates CNN_V1 architecture (CRITICAL STEP)
# 2. Loads trained weights from cnn_v1.pt
# 3. Ensures no architecture mismatch occurs

model = CNN_V1()

state_dict = torch.load("../models/cnn_v1.pt", map_location="cpu")

model.load_state_dict(state_dict)

model.eval()

print("✔ CNN_V1 loaded successfully from frozen .pt file")

✔ CNN_V1 loaded successfully from frozen .pt file


In [8]:
# This cell:
# 1. Creates dummy CIFAR-10-like input
# 2. Runs forward pass through frozen model
# 3. Validates output shape correctness

x = torch.randn(4, 3, 32, 32)

with torch.no_grad():
    output = model(x)

print("Output shape:", output.shape)
print("Expected shape: [4, 10]")

Output shape: torch.Size([4, 10])
Expected shape: [4, 10]


In [10]:
# This cell:
# 1. Runs inference on a random input
# 2. Converts logits to class index
# 3. Maps index to CIFAR-10 class name

import torch

classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

x = torch.randn(1, 3, 32, 32)

with torch.no_grad():
    output = model(x)
    predicted_class = torch.argmax(output, dim=1).item()

print("Predicted class index:", predicted_class)
print("Predicted class name:", classes[predicted_class])

Predicted class index: 6
Predicted class name: frog


In [11]:
# This cell:
# 1. Uses inference module
# 2. Loads frozen CNN_V1 model
# 3. Computes test accuracy from .pt file

from src.inference import run_frozen_model_evaluation
from src.models import CNN_V1

acc = run_frozen_model_evaluation(
    CNN_V1,
    "../models/cnn_v1.pt"
)

print("CNN_V1 Test Accuracy:", round(acc, 4))

CNN_V1 Test Accuracy: 0.7562


In [12]:
# This cell:
# 1. Uses model registry to instantiate CNN_V1
# 2. Loads frozen weights (cnn_v1.pt)
# 3. Evaluates model using inference module
# 4. Prints final test accuracy

from src.model_registry import get_model
from src.inference import evaluate_model
from src.data_loader import get_cifar10_dataloaders
import torch

# -------------------------
# DEVICE
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# LOAD MODEL VIA REGISTRY
# -------------------------
model = get_model("cnn_v1").to(device)

# -------------------------
# LOAD WEIGHTS
# -------------------------
state_dict = torch.load("../models/cnn_v1.pt", map_location=device)
model.load_state_dict(state_dict)

model.eval()

# -------------------------
# DATA
# -------------------------
_, testloader = get_cifar10_dataloaders(batch_size=64)

# -------------------------
# EVALUATION
# -------------------------
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = correct / total

print("CNN_V1 FINAL TEST ACCURACY:", round(acc, 4))

CNN_V1 FINAL TEST ACCURACY: 0.7562
